# Robust LightGCN with Features — Multi-Quarter Sweep (v3)

**v3 = v2 + true out-of-sample (next-quarter) evaluation.**

v2 trained on quarter Q and evaluated on a random 10% slice of Q itself — that measures fit, not forecast. v3 keeps v2's training but additionally:

- For each quarter Q, scores **Q+1's actual Δ-edges** using Q's trained model (restricted to the funds/stocks that exist in both Q and Q+1).
- Reports **OOS AUC / Hit@K / NDCG@K** — link prediction performance on next quarter, never seen during training.
- Computes **Spearman corr(mean_score @ Q, log_return @ Q+1)** — does the model's per-stock rank actually predict next-quarter returns?
- Tags every row in `cusip_ranks_v3` with `predicts_year, predicts_quarter` so the parquet is unambiguous about what each rank is forecasting.

**v2 in-sample metrics are still saved** (under their original column names) as a sanity check / regression baseline.

## 1. Configuration

In [ ]:
EDGES_COLUMN = "change_in_weight"  # or "change_in_adjusted_weight"
DATA_DIR = "../../data"
RESULTS_DIR = "/results/LightGcnV3"

EMBEDDING_DIM = 128
NUM_LAYERS = 3
EPOCHS = 300
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
L2_EMB = 1e-5
EARLY_STOPPING_PATIENCE = 25

BATCH_SIZE = 16384
NUM_NEGATIVES = 5

TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1
RANDOM_SEED = 42
TOP_K_VALUES = [5, 10, 20, 50]

MODEL_TAG = "WeightedLightGCN_v3"
QUARTERS_OVERRIDE = None

## 2. Imports

In [ ]:
import os, time, random, warnings, traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    precision_recall_curve, average_precision_score,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch: {torch.__version__}  device: {DEVICE}")

## 3. Load parquets

In [ ]:
DATA_PATH = Path(DATA_DIR).expanduser().resolve()
if not DATA_PATH.is_dir():
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_PATH}")

RESULTS_PATH = Path(RESULTS_DIR).expanduser().resolve()
RESULTS_PATH.mkdir(parents=True, exist_ok=True)
print(f"DATA_PATH    : {DATA_PATH}")
print(f"RESULTS_PATH : {RESULTS_PATH}")

def _read_parquet(name):
    p = DATA_PATH / f"{name}.parquet"
    df = pd.read_parquet(p) if p.exists() else pd.DataFrame()
    print(f"  loaded {name:30s} {len(df):>10,} rows")
    return df

CHANGED_HOLDINGS = _read_parquet("changed_holdings")
STOCKS_RETURN    = _read_parquet("stocks_return")
CIK_AUM          = _read_parquet("cik_aum")
NORM_HOLDINGS    = _read_parquet("normalized_holdings")
CUSIP_FIN        = _read_parquet("cusip_financial_data")

## 4. Loaders, graph builder, model (same as v2)

In [ ]:
STOCK_FEATURE_COLS = [
    "diluted_eps", "roe", "ev_ebitda", "pe_ratio", "price_to_sales",
    "price_to_book", "debt_to_equity", "dividend_yield", "fcf_per_share",
    "log_return",
]

def next_year_quarter(y, q): return (y + 1, 1) if q == 4 else (y, q + 1)
def prev_year_quarter(y, q): return (y - 1, 4) if q == 1 else (y, q - 1)

def load_edges(year, quarter, col):
    df = CHANGED_HOLDINGS
    if df.empty or col not in df.columns:
        return pd.DataFrame(columns=["cik", "cusip", "w"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & df[col].notna()
    return (df.loc[mask, ["cik", "cusip", col]]
              .rename(columns={col: "w"}).reset_index(drop=True))

def load_returns(year, quarter):
    df = STOCKS_RETURN
    if df.empty:
        return pd.DataFrame(columns=["cusip", "log_return"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & df["log_return"].notna()
    return df.loc[mask, ["cusip", "log_return"]].reset_index(drop=True)

def load_aum(year, quarter):
    df = CIK_AUM
    if df.empty:
        return pd.DataFrame(columns=["cik", "aum"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & (df["total"] > 0)
    return (df.loc[mask, ["cik", "total"]]
              .rename(columns={"total": "aum"}).reset_index(drop=True))

def load_stock_features(year, quarter):
    fin = CUSIP_FIN
    if fin.empty:
        fin = pd.DataFrame(columns=["cusip"] + STOCK_FEATURE_COLS)
    else:
        fin = fin.loc[(fin["year"] == year) & (fin["quarter"] == quarter)].copy()
    rets = load_returns(year, quarter)
    df = fin.merge(rets, on="cusip", how="outer")
    for c in STOCK_FEATURE_COLS:
        if c not in df.columns:
            df[c] = 0.0
    return df[["cusip"] + STOCK_FEATURE_COLS]

def investor_profitability(year, quarter):
    ny, nq = next_year_quarter(year, quarter)
    h = NORM_HOLDINGS
    if h.empty:
        return pd.Series(dtype=float, name="profitability")
    h = h.loc[(h["year"] == year) & (h["quarter"] == quarter), ["cik", "cusip", "weight"]]
    r = load_returns(ny, nq)
    m = h.merge(r, on="cusip", how="inner")
    m["contrib"] = m["weight"] * m["log_return"]
    return m.groupby("cik")["contrib"].sum().rename("profitability")

def zscore(df, cols):
    out = df.copy()
    for c in cols:
        v = out[c].astype(float).replace([np.inf, -np.inf], np.nan)
        v = v.fillna(v.median() if v.notna().any() else 0.0)
        sd = v.std()
        out[c] = (v - v.mean()) / sd if sd > 0 else 0.0
    return out

In [ ]:
def build_feature_graph(year, quarter, col):
    edges = load_edges(year, quarter, col)
    if edges.empty:
        raise RuntimeError(f"no \u0394-edges for {year} Q{quarter}")
    aum = load_aum(year, quarter)
    py, pq = prev_year_quarter(year, quarter)
    try:
        past_prof = investor_profitability(py, pq).reset_index()
    except Exception:
        past_prof = pd.DataFrame(columns=["cik", "profitability"])

    cik_nh = edges.groupby("cik").size().rename("n_holdings").reset_index()
    cik_df = aum.merge(cik_nh, on="cik", how="outer").merge(past_prof, on="cik", how="left")
    cik_df["aum"] = cik_df["aum"].fillna(
        cik_df["aum"].median() if cik_df["aum"].notna().any() else 0.0)
    cik_df["log_aum"] = np.log(cik_df["aum"].clip(lower=1.0))
    cik_df["n_holdings"] = cik_df["n_holdings"].fillna(0)
    cik_df["profitability"] = cik_df["profitability"].fillna(0.0)
    CIK_FEATS = ["log_aum", "n_holdings", "profitability"]
    cik_df = zscore(cik_df, CIK_FEATS)
    stock_df = zscore(load_stock_features(year, quarter), STOCK_FEATURE_COLS)

    cik_ids = pd.Index(edges["cik"].unique())
    cusip_ids = pd.Index(edges["cusip"].unique())
    cik_df = cik_df.set_index("cik").reindex(cik_ids).fillna(0.0)
    stock_df = stock_df.set_index("cusip").reindex(cusip_ids).fillna(0.0)
    F_cik = cik_df[CIK_FEATS].to_numpy()
    F_stk = stock_df[STOCK_FEATURE_COLS].to_numpy()
    Fdim = F_cik.shape[1] + F_stk.shape[1]
    X = np.zeros((len(cik_ids) + len(cusip_ids), Fdim), dtype=np.float32)
    X[:len(cik_ids), :F_cik.shape[1]] = F_cik
    X[len(cik_ids):, F_cik.shape[1]:] = F_stk
    cik_pos = {c: i for i, c in enumerate(cik_ids)}
    cusip_pos = {c: i + len(cik_ids) for i, c in enumerate(cusip_ids)}

    edges = edges.copy()
    edges["src"] = edges["cik"].map(cik_pos).astype(np.int64)
    edges["dst"] = edges["cusip"].map(cusip_pos).astype(np.int64)
    return {
        "X": X,
        "edges": edges[["src", "dst", "w"]].reset_index(drop=True),
        "n_cik": len(cik_ids),
        "n_cusip": len(cusip_ids),
        "n_nodes": len(cik_ids) + len(cusip_ids),
        "cik_ids": cik_ids,
        "cusip_ids": cusip_ids,
        "cik_pos": cik_pos,
        "cusip_pos": cusip_pos,
    }

def split_edges(edges_df, tr, va, te, seed):
    df = edges_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    n = len(df); n_tr = int(n * tr); n_va = int(n * va)
    train = df.iloc[:n_tr].reset_index(drop=True)
    val   = df.iloc[n_tr:n_tr + n_va].reset_index(drop=True)
    test  = df.iloc[n_tr + n_va:].reset_index(drop=True)
    train_nodes = set(train["src"]).union(train["dst"])
    def reflow(part):
        m = ~part["src"].isin(train_nodes) | ~part["dst"].isin(train_nodes)
        return part[~m].reset_index(drop=True), part[m]
    val,  vm = reflow(val)
    test, tm = reflow(test)
    if len(vm) + len(tm) > 0:
        train = pd.concat([train, vm, tm], ignore_index=True)
    return train, val, test

def edges_to_index(df, device):
    src = df["src"].to_numpy(); dst = df["dst"].to_numpy()
    w = df["w"].to_numpy().astype(np.float32)
    ei = np.stack([np.concatenate([src, dst]), np.concatenate([dst, src])], axis=0)
    ew = np.concatenate([np.abs(w), np.abs(w)])
    return (torch.tensor(ei, dtype=torch.long, device=device),
            torch.tensor(ew, dtype=torch.float, device=device))

In [ ]:
def build_forbid_set(edges_df):
    return set(zip(edges_df["src"].astype(int).tolist(),
                   edges_df["dst"].astype(int).tolist()))

def sample_negatives_batch(num_pos, n_cik, n_cusip, num_negatives, forbid, rng):
    target = num_pos * num_negatives
    out = np.empty((target, 2), dtype=np.int64)
    filled = 0
    while filled < target:
        need = target - filled
        m = max(need * 2, 4096)
        u = rng.integers(0, n_cik, size=m)
        v = rng.integers(n_cik, n_cik + n_cusip, size=m)
        keep_u, keep_v = [], []
        for a, b in zip(u, v):
            if (int(a), int(b)) not in forbid:
                keep_u.append(a); keep_v.append(b)
                if len(keep_u) >= need:
                    break
        k = len(keep_u)
        out[filled:filled + k, 0] = keep_u
        out[filled:filled + k, 1] = keep_v
        filled += k
    return out

class WeightedLightGCN(nn.Module):
    def __init__(self, in_feats, embedding_dim, num_layers):
        super().__init__()
        self.input_proj = nn.Linear(in_feats, embedding_dim)
        nn.init.normal_(self.input_proj.weight, std=0.1)
        nn.init.zeros_(self.input_proj.bias)
        self.convs = nn.ModuleList([
            GCNConv(embedding_dim, embedding_dim,
                    improved=False, cached=False, add_self_loops=True)
            for _ in range(num_layers)
        ])
    def forward(self, x, edge_index, edge_weight):
        x = self.input_proj(x)
        layers = [x]
        for conv in self.convs:
            x = conv(x, edge_index, edge_weight=edge_weight)
            layers.append(x)
        return torch.stack(layers, dim=0).mean(dim=0)

def bpr_loss_with_l2(pos_u, pos_v, neg_u, neg_v, l2_emb):
    pos_scores = (pos_u * pos_v).sum(dim=1)
    neg_scores = (neg_u * neg_v).sum(dim=1)
    bpr = -F.logsigmoid(pos_scores - neg_scores).mean()
    reg = (pos_u.pow(2).sum() + pos_v.pow(2).sum()
           + neg_u.pow(2).sum() + neg_v.pow(2).sum()) / pos_u.size(0)
    return bpr + l2_emb * reg, bpr.item()

In [ ]:
def train_model(model, X, train_ei, train_ew, trainval_ei, trainval_ew,
                train_pos_np, val_pos_np,
                n_cik, n_cusip,
                forbid_train, forbid_trainval,
                epochs, lr, weight_decay, l2_emb, patience,
                batch_size, num_negatives, seed):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    rng = np.random.default_rng(seed)
    n_train = len(train_pos_np); n_val = len(val_pos_np)
    train_pos_t = torch.tensor(train_pos_np, dtype=torch.long, device=X.device)
    val_pos_t   = torch.tensor(val_pos_np,   dtype=torch.long, device=X.device)
    train_losses, val_losses = [], []
    best_val = float("inf"); best_state = None; no_improve = 0
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n_train, device=X.device)
        epoch_loss_sum = 0.0; n_batches = 0
        for start in range(0, n_train, batch_size):
            idx = perm[start:start + batch_size]
            pos_batch = train_pos_t[idx]
            B = pos_batch.size(0)
            neg_np = sample_negatives_batch(B, n_cik, n_cusip, num_negatives,
                                            forbid_train, rng)
            neg_batch = torch.tensor(neg_np, dtype=torch.long, device=X.device)
            pos_rep = pos_batch.repeat_interleave(num_negatives, dim=0)
            opt.zero_grad()
            emb = model(X, train_ei, train_ew)
            pu = emb[pos_rep[:, 0]]; pv = emb[pos_rep[:, 1]]
            nu = emb[neg_batch[:, 0]]; nv = emb[neg_batch[:, 1]]
            loss, bpr_val = bpr_loss_with_l2(pu, pv, nu, nv, l2_emb)
            loss.backward(); opt.step()
            epoch_loss_sum += bpr_val; n_batches += 1
        train_losses.append(epoch_loss_sum / max(n_batches, 1))
        model.eval()
        with torch.no_grad():
            v_neg_np = sample_negatives_batch(n_val, n_cik, n_cusip, 1,
                                              forbid_trainval, rng)
            v_neg = torch.tensor(v_neg_np, dtype=torch.long, device=X.device)
            emb = model(X, trainval_ei, trainval_ew)
            vps = (emb[val_pos_t[:, 0]] * emb[val_pos_t[:, 1]]).sum(dim=1)
            vns = (emb[v_neg[:, 0]] * emb[v_neg[:, 1]]).sum(dim=1)
            v_loss = -F.logsigmoid(vps - vns).mean().item()
        val_losses.append(v_loss)
        if v_loss < best_val - 1e-5:
            best_val = v_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, train_losses, val_losses

## 5. Evaluation helpers (in-sample, same as v2)

In [ ]:
def evaluate_test(model, X, edge_index, edge_weight, pos_pairs, neg_pairs,
                  threshold=0.5):
    model.eval()
    with torch.no_grad():
        emb = model(X, edge_index, edge_weight)
        pu = emb[pos_pairs[:,0]]; pv = emb[pos_pairs[:,1]]
        nu = emb[neg_pairs[:,0]]; nv = emb[neg_pairs[:,1]]
        ps = (pu * pv).sum(dim=1).sigmoid().cpu().numpy()
        ns = (nu * nv).sum(dim=1).sigmoid().cpu().numpy()
    scores = np.concatenate([ps, ns])
    labels = np.concatenate([np.ones(len(ps)), np.zeros(len(ns))])
    auc = roc_auc_score(labels, scores)
    pred = (scores > threshold).astype(int)
    return {
        "auc": auc,
        "precision": precision_score(labels, pred, zero_division=0),
        "recall":    recall_score(labels, pred, zero_division=0),
        "f1":        f1_score(labels, pred, zero_division=0),
        "scores": scores, "labels": labels,
    }

def find_optimal_threshold(scores, labels):
    p, r, ths = precision_recall_curve(labels, scores)
    f1 = 2 * p * r / (p + r + 1e-10)
    idx = int(np.argmax(f1))
    th = ths[idx] if idx < len(ths) else 0.5
    return float(th), float(average_precision_score(labels, scores))

def evaluate_top_k(model, X, edge_index, edge_weight, test_edges_df,
                   seen_edges_df, n_cik, n_cusip, k_list):
    """Top-K hit rate / NDCG. Stocks already in `seen_edges_df` for each fund
    are masked (set to -inf) so the top-K reflects genuinely new predictions."""
    model.eval()
    with torch.no_grad():
        emb = model(X, edge_index, edge_weight)
    fund_to_test = {}
    for src, dst in zip(test_edges_df["src"].to_numpy(),
                        test_edges_df["dst"].to_numpy()):
        fund_to_test.setdefault(int(src), set()).add(int(dst))
    fund_to_seen = {}
    for src, dst in zip(seen_edges_df["src"].to_numpy(),
                        seen_edges_df["dst"].to_numpy()):
        fund_to_seen.setdefault(int(src), set()).add(int(dst))
    stock_indices = np.arange(n_cik, n_cik + n_cusip)
    stock_emb = emb[stock_indices]
    res = {k: {"hit_rate": 0.0, "ndcg": 0.0} for k in k_list}
    n_funds = len(fund_to_test)
    for fund_idx, true_stocks in fund_to_test.items():
        scores = (emb[fund_idx] * stock_emb).sum(dim=1).cpu().numpy()
        seen = fund_to_seen.get(fund_idx, set())
        if seen:
            seen_local = np.fromiter(
                (s - n_cik for s in seen if n_cik <= s < n_cik + n_cusip),
                dtype=np.int64,
            )
            if seen_local.size:
                scores[seen_local] = -np.inf
        order = np.argsort(-scores)
        for k in k_list:
            top_k_global = stock_indices[order[:k]]
            hits = [int(s) for s in top_k_global if int(s) in true_stocks]
            if hits:
                res[k]["hit_rate"] += 1
                dcg = sum(
                    1.0 / np.log2(i + 2)
                    for i, s in enumerate(top_k_global) if int(s) in true_stocks)
                idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(true_stocks), k)))
                res[k]["ndcg"] += dcg / idcg if idcg > 0 else 0.0
    for k in k_list:
        res[k]["hit_rate"] /= max(n_funds, 1)
        res[k]["ndcg"]     /= max(n_funds, 1)
    return res

## 6. NEW: out-of-sample (next-quarter) evaluation

Given Q's trained model, evaluate it on **Q+1's actual Δ-edges**:

1. **Restrict to shared universe**: only funds and stocks that exist in both Q and Q+1 (funds/stocks new to Q+1 have no learned embedding).
2. **Score Q+1 positives**: dot-product of Q's embeddings for the shared (fund, stock) pairs that actually appear in Q+1's Δ-graph.
3. **Sample negatives** from Q's universe, forbidding both Q's known edges and Q+1's positives.
4. **Top-K** with Q's edges masked (so we only count *new* recommendations).

Plus: **Spearman corr(mean_score @ Q, log_return @ Q+1)** — does the per-stock score actually predict next-quarter return?

In [ ]:
def compute_stock_ranking(model, X, edge_index, edge_weight,
                          n_cik, n_cusip, cusip_ids, year, quarter,
                          predicts_year, predicts_quarter,
                          score_chunk_funds=2048):
    """Per-stock mean attractiveness score from Q's trained model.
    Tagged with the quarter being PREDICTED (= Q+1)."""
    model.eval()
    with torch.no_grad():
        emb = model(X, edge_index, edge_weight)
        fund_emb  = emb[:n_cik]
        stock_emb = emb[n_cik:n_cik + n_cusip]
        accum = torch.zeros(n_cusip, device=emb.device)
        for start in range(0, n_cik, score_chunk_funds):
            end = min(start + score_chunk_funds, n_cik)
            block = torch.sigmoid(fund_emb[start:end] @ stock_emb.T)
            accum += block.sum(dim=0)
        mean_score = (accum / max(n_cik, 1)).cpu().numpy()
    df = pd.DataFrame({
        "cusip":   list(cusip_ids),
        "year":    int(year),
        "quarter": int(quarter),
        "predicts_year":    int(predicts_year),
        "predicts_quarter": int(predicts_quarter),
        "mean_score": mean_score,
    }).sort_values("mean_score", ascending=False).reset_index(drop=True)
    df["rank"] = np.arange(1, len(df) + 1, dtype=np.int64)
    return df


def evaluate_next_quarter(model, X, full_ei, full_ew, G,
                          next_year, next_quarter, edges_col,
                          k_list, seed):
    """Out-of-sample link prediction on Q+1's Δ-edges.
    Returns a flat dict (NaN-filled if Q+1 not available)."""
    nan_result = {
        "oos_has_next": False, "oos_n_pos": 0,
        "oos_n_shared_funds": 0, "oos_n_shared_stocks": 0,
        "oos_auc": float("nan"), "oos_avg_precision": float("nan"),
        "oos_precision_05": float("nan"), "oos_recall_05": float("nan"),
        "oos_f1_05": float("nan"),
        **{f"oos_hit{k}":  float("nan") for k in k_list},
        **{f"oos_ndcg{k}": float("nan") for k in k_list},
    }
    try:
        next_edges_df = load_edges(next_year, next_quarter, edges_col)
    except Exception:
        return nan_result
    if next_edges_df.empty:
        return nan_result

    cik_pos = G["cik_pos"]; cusip_pos = G["cusip_pos"]
    cik_set  = set(cik_pos.keys())
    cusip_set = set(cusip_pos.keys())
    shared_mask = next_edges_df["cik"].isin(cik_set) & next_edges_df["cusip"].isin(cusip_set)
    next_in_Q = next_edges_df[shared_mask].copy()
    if len(next_in_Q) == 0:
        nan_result["oos_has_next"] = True
        return nan_result
    next_in_Q["src"] = next_in_Q["cik"].map(cik_pos).astype(np.int64)
    next_in_Q["dst"] = next_in_Q["cusip"].map(cusip_pos).astype(np.int64)

    # Forbid: Q's edges (already used in propagation) + Q+1's actual positives
    Q_pos_set = build_forbid_set(G["edges"])
    next_pos_set = build_forbid_set(next_in_Q)
    forbid = Q_pos_set | next_pos_set

    rng = np.random.default_rng(seed)
    n_pos = len(next_in_Q)
    neg_np = sample_negatives_batch(n_pos, G["n_cik"], G["n_cusip"], 1, forbid, rng)
    pos_pairs = torch.tensor(next_in_Q[["src", "dst"]].to_numpy(np.int64), device=X.device)
    neg_pairs = torch.tensor(neg_np, device=X.device)

    model.eval()
    with torch.no_grad():
        emb = model(X, full_ei, full_ew)
        ps = (emb[pos_pairs[:, 0]] * emb[pos_pairs[:, 1]]).sum(dim=1).sigmoid().cpu().numpy()
        ns = (emb[neg_pairs[:, 0]] * emb[neg_pairs[:, 1]]).sum(dim=1).sigmoid().cpu().numpy()
    scores = np.concatenate([ps, ns])
    labels = np.concatenate([np.ones(len(ps)), np.zeros(len(ns))])
    auc = roc_auc_score(labels, scores)
    ap  = average_precision_score(labels, scores)
    pred05 = (scores > 0.5).astype(int)

    # Top-K with Q's edges masked
    oos_top_k = evaluate_top_k(model, X, full_ei, full_ew, next_in_Q, G["edges"],
                                G["n_cik"], G["n_cusip"], k_list)

    return {
        "oos_has_next": True,
        "oos_n_pos": int(n_pos),
        "oos_n_shared_funds":  int(next_in_Q["src"].nunique()),
        "oos_n_shared_stocks": int(next_in_Q["dst"].nunique()),
        "oos_auc": float(auc),
        "oos_avg_precision": float(ap),
        "oos_precision_05": float(precision_score(labels, pred05, zero_division=0)),
        "oos_recall_05":    float(recall_score(labels, pred05, zero_division=0)),
        "oos_f1_05":        float(f1_score(labels, pred05, zero_division=0)),
        **{f"oos_hit{k}":  float(oos_top_k[k]["hit_rate"]) for k in k_list},
        **{f"oos_ndcg{k}": float(oos_top_k[k]["ndcg"])     for k in k_list},
    }


def compute_rank_return_corr(rank_df, next_year, next_quarter):
    """Spearman corr between mean_score @ Q and log_return @ Q+1.
    Positive value => higher predicted attractiveness aligns with higher next-quarter return."""
    nxt = load_returns(next_year, next_quarter)
    if nxt.empty:
        return {"rank_return_spearman": float("nan"), "n_rank_return_pairs": 0}
    joined = rank_df[["cusip", "mean_score"]].merge(
        nxt[["cusip", "log_return"]], on="cusip", how="inner")
    if len(joined) < 5:
        return {"rank_return_spearman": float("nan"), "n_rank_return_pairs": int(len(joined))}
    rs = joined["mean_score"].rank()
    rr = joined["log_return"].rank()
    corr = float(rs.corr(rr))
    return {"rank_return_spearman": corr, "n_rank_return_pairs": int(len(joined))}

## 7. Per-quarter runner

In [ ]:
def run_quarter(year, quarter, edges_col):
    t0 = time.perf_counter()
    G = build_feature_graph(year, quarter, edges_col)
    train_e, val_e, test_e = split_edges(
        G["edges"], TRAIN_RATIO, VAL_RATIO, TEST_RATIO, RANDOM_SEED)

    train_ei, train_ew = edges_to_index(train_e, DEVICE)
    trainval_df = pd.concat([train_e, val_e], ignore_index=True)
    tv_ei, tv_ew = edges_to_index(trainval_df, DEVICE)
    full_ei, full_ew = edges_to_index(G["edges"], DEVICE)
    X_t = torch.tensor(G["X"], dtype=torch.float, device=DEVICE)

    train_pos_np = train_e[["src","dst"]].to_numpy(np.int64)
    val_pos_np   = val_e[["src","dst"]].to_numpy(np.int64)

    forbid_train    = build_forbid_set(train_e)
    forbid_trainval = build_forbid_set(trainval_df)
    forbid_all      = build_forbid_set(pd.concat([trainval_df, test_e], ignore_index=True))

    model = WeightedLightGCN(G["X"].shape[1], EMBEDDING_DIM, NUM_LAYERS).to(DEVICE)
    model, train_losses, val_losses = train_model(
        model, X_t, train_ei, train_ew, tv_ei, tv_ew,
        train_pos_np, val_pos_np,
        G["n_cik"], G["n_cusip"],
        forbid_train, forbid_trainval,
        EPOCHS, LEARNING_RATE, WEIGHT_DECAY, L2_EMB, EARLY_STOPPING_PATIENCE,
        BATCH_SIZE, NUM_NEGATIVES, RANDOM_SEED)

    # ---- in-sample test (sanity / regression baseline) ----
    test_pos = torch.tensor(test_e[["src","dst"]].to_numpy(np.int64), device=DEVICE)
    rng = np.random.default_rng(RANDOM_SEED + 2)
    test_neg_np = sample_negatives_batch(len(test_e), G["n_cik"], G["n_cusip"], 1,
                                         forbid_all, rng)
    test_neg = torch.tensor(test_neg_np, device=DEVICE)
    res = evaluate_test(model, X_t, tv_ei, tv_ew, test_pos, test_neg, threshold=0.5)
    opt_th, ap = find_optimal_threshold(res["scores"], res["labels"])
    res_opt = evaluate_test(model, X_t, tv_ei, tv_ew, test_pos, test_neg, threshold=opt_th)
    in_top_k = evaluate_top_k(model, X_t, tv_ei, tv_ew, test_e, trainval_df,
                                G["n_cik"], G["n_cusip"], TOP_K_VALUES)

    # ---- out-of-sample: predict Q+1 ----
    next_y, next_q = next_year_quarter(year, quarter)
    oos = evaluate_next_quarter(model, X_t, full_ei, full_ew, G,
                                 next_y, next_q, edges_col,
                                 TOP_K_VALUES, RANDOM_SEED + 3)

    # ---- per-stock ranking (tagged with predicts_year/quarter = Q+1) ----
    rank_df = compute_stock_ranking(model, X_t, full_ei, full_ew,
                                     G["n_cik"], G["n_cusip"],
                                     G["cusip_ids"], year, quarter,
                                     predicts_year=next_y, predicts_quarter=next_q)

    # ---- rank-vs-return spearman corr ----
    rrc = compute_rank_return_corr(rank_df, next_y, next_q)

    metrics = {
        "year": int(year), "quarter": int(quarter),
        "predicts_year": int(next_y), "predicts_quarter": int(next_q),
        "n_funds": int(G["n_cik"]), "n_stocks": int(G["n_cusip"]),
        "n_edges": int(len(G["edges"])),
        "n_train": int(len(train_e)), "n_val": int(len(val_e)),
        "n_test":  int(len(test_e)),
        "epochs_trained": int(len(train_losses)),
        "final_train_loss": float(train_losses[-1]),
        "final_val_loss":   float(val_losses[-1]),
        "best_val_loss":    float(min(val_losses)),
        # in-sample
        "auc":           float(res["auc"]),
        "avg_precision": float(ap),
        "opt_threshold": float(opt_th),
        "precision_05":  float(res["precision"]),
        "recall_05":     float(res["recall"]),
        "f1_05":         float(res["f1"]),
        "precision_opt": float(res_opt["precision"]),
        "recall_opt":    float(res_opt["recall"]),
        "f1_opt":        float(res_opt["f1"]),
        **{f"hit{k}":  float(in_top_k[k]["hit_rate"]) for k in TOP_K_VALUES},
        **{f"ndcg{k}": float(in_top_k[k]["ndcg"])     for k in TOP_K_VALUES},
        # out-of-sample
        **oos,
        # rank-return
        **rrc,
        "elapsed_s": float(time.perf_counter() - t0),
    }

    del model, X_t, train_ei, train_ew, tv_ei, tv_ew, full_ei, full_ew
    del test_pos, test_neg
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics, rank_df

## 8. Resumable I/O (atomic writes)

In [ ]:
RESULTS_CSV   = RESULTS_PATH / f"sweep_results_v3__{EDGES_COLUMN}.csv"
RANKS_PARQUET = RESULTS_PATH / f"cusip_ranks_v3__{EDGES_COLUMN}.parquet"
print(f"results CSV    : {RESULTS_CSV}")
print(f"ranks parquet  : {RANKS_PARQUET}")

def list_available_quarters(edges_col):
    df = CHANGED_HOLDINGS
    sub = df.loc[df[edges_col].notna(), ["year", "quarter"]].drop_duplicates()
    yq = sorted({(int(y), int(q)) for y, q in sub.itertuples(index=False)})
    avail = set(yq)
    return [(y, q) for (y, q) in yq if prev_year_quarter(y, q) in avail]

def load_done_set(csv_path):
    if not csv_path.exists():
        return set()
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"  [warn] could not parse {csv_path.name}: {e!r}; ignoring")
        return set()
    if "year" not in df.columns or "quarter" not in df.columns:
        return set()
    return set(zip(df["year"].astype(int), df["quarter"].astype(int)))

def _atomic_write(write_fn, target_path):
    target_path = Path(target_path)
    tmp = target_path.with_name(target_path.name + ".tmp")
    try:
        write_fn(tmp)
        os.replace(tmp, target_path)
    finally:
        if tmp.exists():
            try: tmp.unlink()
            except OSError: pass

def append_metrics_row(row, csv_path):
    if csv_path.exists():
        try:
            existing = pd.read_csv(csv_path)
        except Exception as e:
            print(f"  [warn] could not parse {csv_path.name} ({e!r}); backing up")
            os.replace(csv_path, csv_path.with_suffix(".csv.corrupt"))
            existing = pd.DataFrame()
    else:
        existing = pd.DataFrame()
    df = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
    _atomic_write(lambda p: df.to_csv(p, index=False), csv_path)

def append_ranks(rank_df, parquet_path, model_tag):
    rank_df = rank_df.copy()
    rank_df["model"] = model_tag
    if parquet_path.exists():
        try:
            prev = pd.read_parquet(parquet_path)
            y = int(rank_df["year"].iloc[0]); q = int(rank_df["quarter"].iloc[0])
            prev = prev[~((prev["year"] == y) & (prev["quarter"] == q) &
                          (prev["model"] == model_tag))]
            rank_df = pd.concat([prev, rank_df], ignore_index=True)
        except Exception as e:
            print(f"  [warn] could not parse {parquet_path.name} ({e!r}); backing up")
            os.replace(parquet_path, parquet_path.with_suffix(".parquet.corrupt"))
    _atomic_write(lambda p: rank_df.to_parquet(p, index=False), parquet_path)

## 9. Sweep

Trains one quarter at a time, appends to CSV + parquet after each one. Re-running this cell resumes from the last completed quarter.

In [ ]:
all_quarters = list_available_quarters(EDGES_COLUMN)
if QUARTERS_OVERRIDE is not None:
    requested = [(int(y), int(q)) for y, q in QUARTERS_OVERRIDE]
    quarters = [yq for yq in requested if yq in set(all_quarters)]
    skipped = [yq for yq in requested if yq not in set(all_quarters)]
    if skipped:
        print(f"  [skip] not in available quarters: {skipped}")
else:
    quarters = all_quarters

done = load_done_set(RESULTS_CSV)
remaining = [yq for yq in quarters if yq not in done]
print(f"available quarters: {len(all_quarters)}  selected: {len(quarters)}  done: {len(done)}  remaining: {len(remaining)}")
if remaining:
    print(f"  range: {remaining[0]} .. {remaining[-1]}")

t_start = time.perf_counter()
failures = []
for i, (y, q) in enumerate(remaining, 1):
    try:
        metrics, rank_df = run_quarter(y, q, EDGES_COLUMN)
        append_metrics_row(metrics, RESULTS_CSV)
        append_ranks(rank_df, RANKS_PARQUET, MODEL_TAG)
        elapsed = time.perf_counter() - t_start
        eta = elapsed / i * max(len(remaining) - i, 0)
        oos_str = (f"oos_auc={metrics['oos_auc']:.4f} oos_hit10={metrics['oos_hit10']:.4f} "
                   f"rrc={metrics['rank_return_spearman']:.3f}"
                   if metrics.get("oos_has_next") else "oos=N/A")
        print(f"[{len(done)+i:2d}/{len(quarters)}] {y} Q{q}  "
              f"in_auc={metrics['auc']:.4f}  {oos_str}  "
              f"({metrics['epochs_trained']:>3}ep, {metrics['elapsed_s']:.0f}s)  "
              f"ETA {eta/60:.1f}m")
    except Exception as e:
        failures.append((y, q, repr(e)))
        print(f"  ! {y} Q{q} FAILED: {e.__class__.__name__}: {e}")
        traceback.print_exc()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

total_min = (time.perf_counter() - t_start) / 60
print(f"\nsweep complete in {total_min:.1f} min  ({len(remaining) - len(failures)} ok, {len(failures)} failed)")
if failures:
    print("failed quarters:")
    for f in failures:
        print(f"  {f}")

## 10. Inspect results

In [ ]:
if RESULTS_CSV.exists():
    res_df = pd.read_csv(RESULTS_CSV).sort_values(["year", "quarter"]).reset_index(drop=True)
    print(f"quarters in results: {len(res_df)}")
    cols = ["year", "quarter", "predicts_year", "predicts_quarter",
            "auc", "hit10", "oos_auc", "oos_hit10", "oos_ndcg10",
            "rank_return_spearman", "n_rank_return_pairs", "epochs_trained"]
    cols = [c for c in cols if c in res_df.columns]
    print(res_df[cols].to_string(index=False))
    print()
    print("Aggregates (mean \u00b1 std):")
    agg_cols = ["auc", "hit10", "oos_auc", "oos_hit10", "oos_ndcg10",
                "rank_return_spearman"]
    agg_cols = [c for c in agg_cols if c in res_df.columns]
    print(res_df[agg_cols].agg(["mean", "std"]).T.round(4))
else:
    print("no results yet")

In [ ]:
# Plot in-sample vs out-of-sample AUC + rank-return correlation over time
if RESULTS_CSV.exists() and len(res_df) > 0:
    res_df["yq"] = res_df["year"] + (res_df["quarter"] - 1) / 4
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    axes[0].plot(res_df["yq"], res_df["auc"], marker="o", color="steelblue", label="in-sample")
    if "oos_auc" in res_df.columns:
        axes[0].plot(res_df["yq"], res_df["oos_auc"], marker="s", color="darkorange", label="out-of-sample (Q+1)")
    axes[0].axhline(0.5, color="gray", linestyle="--", alpha=0.5)
    axes[0].set_xlabel("Year"); axes[0].set_ylabel("AUC-ROC")
    axes[0].set_title("AUC: training quarter vs next quarter")
    axes[0].grid(True); axes[0].legend()

    if "oos_hit10" in res_df.columns:
        axes[1].plot(res_df["yq"], res_df["hit10"],     marker="o", label="in-sample hit@10")
        axes[1].plot(res_df["yq"], res_df["oos_hit10"], marker="s", label="oos hit@10")
        axes[1].plot(res_df["yq"], res_df["oos_ndcg10"], marker="^", label="oos ndcg@10")
    axes[1].set_xlabel("Year"); axes[1].set_ylabel("score")
    axes[1].set_title("Top-10 ranking quality")
    axes[1].grid(True); axes[1].legend()

    if "rank_return_spearman" in res_df.columns:
        axes[2].axhline(0, color="gray", linestyle="--", alpha=0.5)
        axes[2].plot(res_df["yq"], res_df["rank_return_spearman"],
                     marker="o", color="crimson")
        axes[2].set_xlabel("Year"); axes[2].set_ylabel("Spearman corr")
        axes[2].set_title("corr(mean_score @ Q, log_return @ Q+1)")
        axes[2].grid(True)

    plt.tight_layout(); plt.show()

In [ ]:
# Top stocks predicted for the latest available quarter
if RANKS_PARQUET.exists():
    ranks = pd.read_parquet(RANKS_PARQUET)
    print(f"total ranking rows: {len(ranks):,}")
    print(f"quarters with ranks: {ranks[['year','quarter']].drop_duplicates().shape[0]}")
    print()
    last_y = ranks["year"].max()
    last_q = ranks.loc[ranks["year"] == last_y, "quarter"].max()
    latest = ranks[(ranks["year"] == last_y) & (ranks["quarter"] == last_q)]
    pred_y = int(latest["predicts_year"].iloc[0])
    pred_q = int(latest["predicts_quarter"].iloc[0])
    print(f"Top 20 stocks ranked from {last_y} Q{last_q}, predicting {pred_y} Q{pred_q}:")
    print(latest.head(20).to_string(index=False))
else:
    print("no ranks parquet yet")

## 11. Notes

- **In-sample vs out-of-sample.** Columns prefixed `oos_` are computed against **Q+1's actual Δ-edges**, restricted to the (fund, stock) universe shared between Q and Q+1. They are the honest forecasting metrics.
- **Rank-return Spearman.** `rank_return_spearman > 0` means stocks the model ranked higher in Q tended to have higher log_return in Q+1. A consistently positive average across quarters indicates real predictive power.
- **Last quarter has no Q+1.** All `oos_*` and `rank_return_*` columns will be NaN for the most recent quarter (we still save the in-sample row).
- **`cusip_ranks_v3` is unambiguous.** Each row carries both `(year, quarter)` (the quarter the model was trained on) and `(predicts_year, predicts_quarter)` (= Q+1, the quarter being forecast).
- **Resume.** Re-run cell 9 to pick up from the last completed quarter; all writes are atomic (`.tmp` + `os.replace`).